# Домашняя работа: классификация тональности RuTweetCorp

В ноутбуке собраны все этапы работы:

1. загрузка и подготовка данных;
2. baseline на TF-IDF + Logistic Regression;
3. MLP и подбор гиперпараметров;
4. анализ весов признаков;
5. оценка качества, confusion matrix и ручное тестирование.

> Ноутбук рассчитан на запуск из проекта `llm-hw01`, где пакет `llm_hw01`
> доступен в текущем Python-окружении.

In [ ]:
from pathlib import Path
import os
import sys

current_dir = Path.cwd()

project_root = next(
    path
    for path in [current_dir, *current_dir.parents]
    if (path / "pyproject.toml").exists()
)

os.chdir(project_root)

print("Project root:", Path.cwd())
print("Python:", sys.executable)


## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from llm_hw01.dataset import load_dataset
from llm_hw01.preprocessing import (
    clean_text,
    stem_text,
    lemmatize_text,
)

## 1. Загрузка и подготовка датасета


In [ ]:
df = load_dataset()

print("Shape:", df.shape)
print()
print("Class balance:")
print(df["target"].value_counts())
print()
print("Missing values:")
print(df.isna().sum())

df.head()

### Разделение на train/test


In [ ]:
X = df["ttext"]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print("Train:", len(X_train))
print("Test:", len(X_test))

## 2. Task 1 — TF-IDF + Logistic Regression

Сравниваются четыре варианта текста:

- исходный;
- очищенный;
- после stemming;
- после lemmatization.

In [ ]:
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

X_train_stemmed = X_train_clean.apply(stem_text)
X_test_stemmed = X_test_clean.apply(stem_text)

X_train_lemmatized = X_train_clean.apply(lemmatize_text)
X_test_lemmatized = X_test_clean.apply(lemmatize_text)

In [ ]:
def evaluate_logreg(name, X_train_variant, X_test_variant):
    model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                max_features=10000,
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
            ),
        ),
    ])

    model.fit(X_train_variant, y_train)
    pred = model.predict(X_test_variant)

    accuracy = accuracy_score(y_test, pred)

    print(f"{name}: {accuracy:.6f}")
    return accuracy


task1_results = {
    "raw": evaluate_logreg("RAW", X_train, X_test),
    "clean": evaluate_logreg("CLEAN", X_train_clean, X_test_clean),
    "stemmed": evaluate_logreg("STEMMED", X_train_stemmed, X_test_stemmed),
    "lemmatized": evaluate_logreg(
        "LEMMATIZED",
        X_train_lemmatized,
        X_test_lemmatized,
    ),
}

pd.Series(task1_results, name="accuracy").sort_values(ascending=False)

### Результаты

| Вариант | Accuracy |
|---|---:|
| Raw text | **0.74724** |
| Cleaned | 0.74014 |
| Cleaned + Stemming | 0.74554 |
| Cleaned + Lemmatization | 0.74473 |

### Classification report

| Вариант | Класс | Precision | Recall | F1-score |
|---|---:|---:|---:|---:|
| Raw | 0 | 0.75 | 0.73 | 0.74 |
| Raw | 1 | 0.74 | 0.77 | 0.75 |
| Cleaned | 0 | 0.75 | 0.71 | 0.73 |
| Cleaned | 1 | 0.73 | 0.77 | 0.75 |
| Stemming | 0 | 0.75 | 0.72 | 0.74 |
| Stemming | 1 | 0.74 | 0.77 | 0.75 |
| Lemmatization | 0 | 0.75 | 0.72 | 0.74 |
| Lemmatization | 1 | 0.74 | 0.77 | 0.75 |

### Вывод

Лучший результат показала модель на исходном тексте:

**Accuracy = 0.74724**

Простая очистка текста снизила accuracy до `0.74014`. Это показывает, что часть удаляемых элементов текста может содержать полезный  сигнал 

нормализация улучшила результат относительно очищенного текста:

- Stemming: `0.74554`
- Lemmatization: `0.74473`

В данном эксперименте stemming показал немного более высокий результат, чем lemmatization с небольшой разницей 

### Обработка хештегов

Сравнил два варианта обработки хештегов:

- полное удаление хештега вместе со словом;
- удаление только символа `#` с сохранением слова.

При полном удалении хештегов были получены результаты:

| Вариант | Accuracy |
|---|---:|
| Cleaned | 0.73866 |
| Cleaned + Stemming | 0.74269 |
| Cleaned + Lemmatization | 0.74308 |

После сохранения содержимого хештегов:

| Вариант | Accuracy |
|---|---:|
| Cleaned | 0.74014 |
| Cleaned + Stemming | **0.74554** |
| Cleaned + Lemmatization | 0.74473 |

Сохранение текста хештега улучшило результат, поэтому в итоговой предобработке удаляется только символ `#`

## 3. Task 2 — MLP и подбор гиперпараметров


In [ ]:
X_train_mlp, X_val, y_train_mlp, y_val = train_test_split(
    X_train_stemmed,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train,
)

print("MLP train:", len(X_train_mlp))
print("Validation:", len(X_val))

### 1. Подбор архитектуры MLP

Параметры TF-IDF: `max_features=10000`, `ngram_range=(1, 2)`.
Оптимизатор: `Adam`, функция активации: `ReLU`, `early_stopping=True`.

| Hidden layers | Validation Accuracy |
|---|---:|
| `(16,)` | **0.74229** |
| `(32,)` | 0.74085 |
| `(64,)` | 0.74051 |
| `(128,)` | 0.73594 |
| `(256,)` | 0.73913 |
| `(64, 64)` | 0.73855 |
| `(128, 64)` | 0.73812 |

Лучший результат на этом этапе: один скрытый слой из 16 нейронов.

### 2. Совместный подбор `max_features` и размера скрытого слоя

Параметры: `ngram_range=(1, 2)`, `activation=ReLU`, `solver=Adam`.

| max_features | `(16,)` | `(32,)` |
|---:|---:|---:|
| 5 000 | 0.73657 | 0.73625 |
| 10 000 | 0.74229 | 0.74085 |
| 20 000 | 0.74588 | **0.74703** |

Лучший результат: `max_features=20000`, `hidden_layer_sizes=(32,)`.

### 3. Подбор `min_df`

Параметры: `max_features=20000`, `hidden_layer_sizes=(32,)`, `ReLU + Adam`.

| min_df | Validation Accuracy |
|---:|---:|
| `1` (default) | **0.74703** |
| `2` | 0.74628 |
| `3` | 0.74634 |
| `5` | 0.74594 |

Фильтрация редких признаков не улучшила результат, поэтому оставлено `min_df=1`.

### 4. Подбор `max_df`

Параметры: `max_features=20000`, `min_df=1`, `hidden_layer_sizes=(32,)`, `ReLU + Adam`.

| max_df | Validation Accuracy |
|---:|---:|
| `1.0` (default) | **0.74703** |
| `0.95` | 0.74703 |
| `0.90` | 0.74703 |

Изменение `max_df` не повлияло на результат.

### 5. Подбор функции активации и оптимизатора

Параметры TF-IDF: `max_features=20000`, `ngram_range=(1, 2)`.  
Архитектура MLP: `hidden_layer_sizes=(32,)`.

| Activation | Solver | Learning rate | Validation Accuracy |
|---|---|---:|---:|
| `relu` | `adam` | `0.001` | 0.74703 |
| `tanh` | `adam` | `0.001` | **0.74749** |
| `relu` | `sgd` | `0.001` | 0.73913 |
| `relu` | `sgd` | `0.01` | 0.74657 |
| `tanh` | `sgd` | `0.01` | 0.74720 |

Лучший результат показала комбинация `tanh + Adam`.

## Итоговая конфигурация

После подбора гиперпараметров была выбрана следующая конфигурация:

- `max_features=20000`
- `ngram_range=(1, 2)`
- `hidden_layer_sizes=(32,)`
- `activation="tanh"`
- `solver="adam"`
- `early_stopping=True`

Результаты:

| Dataset | Accuracy |
|---|---:|
| Validation | 0.74749 |
| Test | **0.74609** |

Разница между validation и test составила около 0.14%
Финальный test classification report:

| Class | Precision | Recall | F1-score |
|---|---:|---:|---:|
| 0 | 0.74 | 0.74 | 0.74 |
| 1 | 0.75 | 0.75 | 0.75 |

Итоговая accuracy на test: **0.74609**.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
)

X_train_tfidf = vectorizer.fit_transform(X_train_stemmed)
X_test_tfidf = vectorizer.transform(X_test_stemmed)

mlp = MLPClassifier(
    hidden_layer_sizes=(32,),
    activation="tanh",
    solver="adam",
    early_stopping=True,
    n_iter_no_change=5,
    max_iter=100,
    random_state=42,
)

mlp.fit(X_train_tfidf, y_train)

y_test_pred = mlp.predict(X_test_tfidf)

print("Test accuracy:", accuracy_score(y_test, y_test_pred))
print("Iterations:", mlp.n_iter_)
print("Final loss:", mlp.loss_)

## 4. Task 3 — анализ весов модели

Первый слой MLP связывает 20 000 TF-IDF-признаков с 32 нейронами скрытого слоя.

Для каждого признака вычисляется приближённый signed score:

`W1 @ W2`

где `W1` — веса `TF-IDF -> hidden`, а `W2` — веса `hidden -> output`.

Так как между слоями используется `tanh`, это не точный глобальный вклад признака,
а приближённая оценка направления влияния.

In [ ]:
feature_names = vectorizer.get_feature_names_out()

first_layer_weights = mlp.coefs_[0]
second_layer_weights = mlp.coefs_[1]

print("Feature names shape:", feature_names.shape)
print("First layer weights shape:", first_layer_weights.shape)
print("Second layer weights shape:", second_layer_weights.shape)

feature_scores = (
    first_layer_weights @ second_layer_weights
)[:, 0]

In [ ]:
negative_indices = np.where(feature_scores < 0)[0]
top_negative_indices = negative_indices[
    np.argsort(feature_scores[negative_indices])[:20]
]

positive_indices = np.where(feature_scores > 0)[0]
top_positive_indices = positive_indices[
    np.argsort(feature_scores[positive_indices])[::-1][:20]
]

print("Top 20 negative features:")
for idx in top_negative_indices:
    print(f"{feature_names[idx]:<25} {feature_scores[idx]:.6f}")

print()

print("Top 20 positive features:")
for idx in top_positive_indices:
    print(f"{feature_names[idx]:<25} {feature_scores[idx]:.6f}")

### Анализ признаков

Позитивные признаки: `приятн`, `обожа`, `рад`, `спасиб`, `счастлив`, `прекрасн`

Негативные признаки: `обидн`, `грустн`, `печальн`, `сожален`, `ужасн`, `жалк`, `скуча`, `ненавиж`

`о_`, `o_o`, `99`, `xd`, `dd` и остальные варианты с `d` — это смайлы/текстовые эмоциональные маркеры, и в целом они также логично разместились по позитивному и негативному влиянию

Признаки `не плох` и `не зря` показывают пользу использования `ngram_range=(1, 2)`

## 5. Task 4 — качество модели и слабые места

In [ ]:
accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(
    y_test,
    y_test_pred,
    average="weighted",
)
recall = recall_score(
    y_test,
    y_test_pred,
    average="weighted",
)
f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted",
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print()
print("Classification Report:")
print(classification_report(y_test, y_test_pred))

В текущем запуске были получены примерно:

- Accuracy: `0.7456`
- Precision (weighted): `0.7459`
- Recall (weighted): `0.7456`
- F1-score (weighted): `0.7456`

In [ ]:
cm = confusion_matrix(
    y_test,
    y_test_pred,
)

tn, fp, fn, tp = cm.ravel()

print(f"False Positive: {fp}")
print(f"False Negative: {fn}")
print(f"True Positive: {tp}")
print(f"True Negative: {tn}")

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"],
)

disp.plot()
plt.title("Confusion Matrix")
plt.show()

![Confusion Matrix](images/confusion_matrix.png)

False Positive: `5230`  
False Negative: `5835`  
True Positive: `16244`  
True Negative: `16179`

Модель немного уходит в False Negative. То есть модель немного чаще ошибочно принимает позитивные тексты за негативные, чем негативные за позитивные. При этом разница невелика.

## 6. Ручное тестирование

Новые предложения проходят через тот же pipeline:

`clean_text -> stem_text -> vectorizer.transform -> mlp.predict`

In [ ]:
test_sentences = [
    "Сегодня прекрасный день, я очень счастлив",
    "Спасибо, мне всё очень понравилось",
    "Я ненавижу этот ужасный день",
    "Мне очень грустно и плохо",
    "Этот фильм совсем не плохой",
    "Как же здорово снова заболеть",
    "Прекрасно, автобус опять опоздал на час",
    "Единая Россия делает вбросы на выборах",
    "Ерундистика какая-то, но мне понравилось",
    "Я не могу поверить, что это произошло, это не прикольно",
]

processed_sentences = []

for sentence in test_sentences:
    cleaned = clean_text(sentence)
    stemmed = stem_text(cleaned)
    processed_sentences.append(stemmed)

manual_tfidf = vectorizer.transform(
    processed_sentences
)

manual_predictions = mlp.predict(
    manual_tfidf
)

manual_probabilities = mlp.predict_proba(
    manual_tfidf
)

for sentence, prediction, probability in zip(
    test_sentences,
    manual_predictions,
    manual_probabilities,
):
    sentiment = (
        "Positive"
        if prediction == 1
        else "Negative"
    )

    print(f"Text: {sentence}")
    print(f"Prediction: {sentiment}")
    print(
        f"Negative: {probability[0]:.4f}, "
        f"Positive: {probability[1]:.4f}"
    )
    print()

### Анализ своих выражений и слабых мест

Модель хорошо классифицирует явные утверждения, где есть яркие эмоциональные маркеры.

С отрицанием модель не всегда справляется адекватно. Это видно на примере `Этот фильм совсем не плохой`: модель определила его как негативное, хотя у нас используется `ngram_range=(1, 2)`.

Сарказм модель не может адекватно понять: она скорее смотрит на отдельные слова-маркеры и их сочетания, не понимая смысл выражения целиком.

Если в предложении есть противоречивые выражения, модель также может неуверенно определять общий оттенок фразы.

## Итог

В ходе работы были проверены разные варианты предобработки текста и параметры TF-IDF/MLP.

Лучший baseline на Logistic Regression получился на исходном тексте. Для MLP после подбора гиперпараметров была выбрана конфигурация с `max_features=20000`, `hidden_layer_sizes=(32,)`, `tanh` и `Adam`.

Анализ весов показал, что среди сильных признаков в основном находятся логичные эмоциональные слова, смайлы и биграммы. При ручном тестировании стало видно, что основной слабостью подхода остаются отрицания, сарказм и предложения с противоречивыми эмоциональными маркерами.
